1. Develop a labelled corpus containing at least 40 queries across 5 or more intent categories.

In [ ]:
import pandas as pd
from IPython.display import display
corpus = {'Symptoms': ['thirst', 'frequent urination', 'blurred vision', 'fatigue', 'unexplained weight loss', 'slow healing', 'tingling feet', 'increased hunger'], 'Types': ['type 1 diabetes', 'type 2 diabetes', 'gestational diabetes', 'prediabetes', 'insulin resistance', 'difference between diabetes types', 'autoimmune diabetes', 'diabetes during pregnancy'], 'Diet': ['diabetes diet', 'healthy meals', 'carbohydrate intake', 'whole grains', 'sugary drinks', 'food choices', 'fiber rich food', 'meal planning'], 'Exercise': ['walking exercise', 'physical activity', 'workout routine', 'cycling benefits', 'exercise safety', 'strength training', 'active lifestyle', 'exercise frequency'], 'Testing': ['blood glucose test', 'HbA1c test', 'fasting sugar test', 'glucose monitoring', 'blood sugar check', 'diabetes screening', 'glucose meter', 'A1c results'], 'Medication': ['insulin treatment', 'metformin medicine', 'diabetes tablets', 'medication side effects', 'prescribed insulin', 'glucose lowering drugs', 'diabetes prescription', 'medicine schedule'], 'Complications': ['diabetic kidney disease', 'nerve damage', 'eye complications', 'foot ulcers', 'diabetic retinopathy', 'neuropathy', 'kidney problems', 'foot infection'], 'Prevention': ['prevent type 2 diabetes', 'reduce diabetes risk', 'family history risk', 'healthy weight', 'diabetes prevention', 'prevent prediabetes progression', 'risk factors', 'prevent insulin resistance']}
rows = [(query, intent) for intent, items in corpus.items() for query in items]
df = pd.DataFrame(rows, columns=['query','intent'])
display(df)
print('Total queries:',len(df))
print('Intent categories:',df['intent'].nunique())

2. Create a hierarchical taxonomy with at least 4 major categories and 2 subcategories under each.

In [ ]:
taxonomy = {'Disease information':['Symptoms','Types'], 'Lifestyle':['Diet','Exercise'], 'Clinical care':['Testing','Medication'], 'Risk management':['Complications','Prevention']}
for category, items in taxonomy.items():
    print(category, '->', ', '.join(items))

3. Identify at least 8 domain concepts and define at least 10 relationships among them.

In [ ]:
ontology = {'Diabetes': {'affects': 'Blood glucose', 'may_cause': 'Neuropathy'}, 'Insulin': {'regulates': 'Blood glucose'}, 'HbA1c': {'measures': 'Long-term glucose'}, 'Exercise': {'supports': 'Glucose control'}, 'Diet': {'influences': 'Blood glucose'}, 'Neuropathy': {'affects': 'Nerves'}, 'Metformin': {'helps_manage': 'Type 2 diabetes'}, 'Screening': {'detects': 'Prediabetes'}, 'Type 2 diabetes': {'involves': 'Insulin resistance'}}
concepts = set(ontology)
relationships = []
for concept, links in ontology.items():
    for relation, target in links.items():
        concepts.add(target)
        relationships.append((concept,relation,target))
display(pd.DataFrame(relationships,columns=['Concept','Relationship','Target']))
print('Concepts:',len(concepts))
print('Relationships:',len(relationships))

4. Implement lowercase conversion, tokenization, stopword removal and lemmatization.

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords',quiet=True)
nltk.download('wordnet',quiet=True)
stop_words = set(stopwords.words('english')) - {'not','no'}
lemmatizer = WordNetLemmatizer()
def preprocess(text):
    words = re.findall(r'[a-z0-9]+',text.lower())
    return ' '.join(lemmatizer.lemmatize(word) for word in words if word not in stop_words)
df['clean_query'] = df['query'].apply(preprocess)
display(df.head())

5. Apply TF-IDF for feature extraction and train a classifier using Naive Bayes, Logistic Regression or SVM.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
X_train,X_test,y_train,y_test = train_test_split(df['clean_query'],df['intent'],test_size=0.25,random_state=42,stratify=df['intent'])
vectorizer = TfidfVectorizer(ngram_range=(1,2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
model = MultinomialNB(alpha=0.1)
model.fit(X_train_vec,y_train)
print('Training samples:',len(X_train))
print('Testing samples:',len(X_test))
print('TF-IDF shape:',X_train_vec.shape)

6. Evaluate the model using accuracy, precision, recall, F1-score and a confusion matrix.

In [ ]:
from sklearn.metrics import accuracy_score,classification_report,ConfusionMatrixDisplay
import matplotlib.pyplot as plt
predictions = model.predict(X_test_vec)
print('Accuracy:',round(accuracy_score(y_test,predictions),4))
print(classification_report(y_test,predictions,zero_division=0))
fig,ax = plt.subplots(figsize=(9,7))
ConfusionMatrixDisplay.from_predictions(y_test,predictions,labels=model.classes_,xticks_rotation=45,ax=ax,cmap='Blues')
plt.tight_layout()
plt.show()

7. Develop a domain-specific knowledge base containing relevant information for each intent category.

In [ ]:
knowledge = {'Symptoms': 'Diabetes can cause thirst, frequent urination and fatigue. Symptoms alone cannot confirm diabetes; consult a clinician for testing.', 'Types': 'Type 1 involves insufficient insulin production; type 2 involves insulin resistance and impaired insulin production. Gestational diabetes occurs during pregnancy.', 'Diet': 'Choose balanced meals with vegetables, whole grains and protein. A dietitian can help tailor carbohydrate intake.', 'Exercise': 'Regular physical activity can support glucose management. Discuss safe activity with your clinician, especially if taking glucose-lowering medicines.', 'Testing': 'Blood glucose and HbA1c tests help assess glucose levels. Ask a clinician to interpret results and recommend monitoring.', 'Medication': 'Diabetes medicines differ by individual needs. Follow your prescription and consult your clinician before changing medication.', 'Complications': 'Diabetes can affect nerves, kidneys, eyes and feet. Regular screening and foot checks are important; seek care for new wounds.', 'Prevention': 'Healthy eating, physical activity and weight management when appropriate can reduce type 2 diabetes risk. Type 1 diabetes is not currently preventable through lifestyle changes.'}
for intent,response in knowledge.items():
    print(intent,':',response)

8. Implement at least 5 rules, including emergency escalation, low-confidence handling and unsupported-query handling.

In [ ]:
emergency_terms = ['chest pain','cannot breathe','can not breathe','difficulty breathing','shortness of breath','fainted','unconscious','seizure','stroke symptoms','severe confusion','face drooping','slurred speech']
unsupported_terms = ['malaria','dengue','fracture','skin rash','toothache','cancer','ear infection','asthma','weather','football']
def apply_rules(query,vector=None,confidence=None):
    text = ' '.join(re.findall(r'[a-z0-9]+',query.lower()))
    if any(term in text for term in emergency_terms):
        return 'Possible emergency. If this is happening now, call local emergency services (112 in India) immediately.'
    if not text:
        return 'Please enter a question.'
    if re.search(r'\b(dose|dosage|double|stop taking|increase medication)\b',text):
        return 'Please ask your doctor or pharmacist before changing any medication.'
    if text in ['hi','hello','hey','good morning']:
        return 'Hello! Ask me a question about the supported health topics.'
    if any(term in text for term in unsupported_terms):
        return 'This question is outside my supported health topics. Please consult a healthcare professional.'
    if vector is not None and vector.nnz == 0:
        return 'I do not recognize this topic. Please ask about the supported health topics.'
    if confidence is not None and confidence < 0.45:
        return 'I am not sure what you mean. Could you rephrase your question with more detail?'
    return None

9. Integrate NLP, intent classification, reasoning and knowledge retrieval into a single assistant.

In [ ]:
def assistant(query):
    response = apply_rules(query)
    if response is not None:
        return response
    vector = vectorizer.transform([preprocess(query)])
    probabilities = model.predict_proba(vector)[0]
    confidence = probabilities.max()
    response = apply_rules(query,vector,confidence)
    if response is not None:
        return response
    intent = model.classes_[probabilities.argmax()]
    return knowledge[intent]

print(assistant('hello'))

10. Deploy the assistant in Google Colab with an interactive interface that accepts multiple queries and displays appropriate responses.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
query_box = widgets.Text(placeholder='Enter your health question',description='You:')
submit = widgets.Button(description='Ask')
output = widgets.Output()
def respond(button):
    query = query_box.value
    with output:
        print('You:',query)
        print('Assistant:',assistant(query))
    query_box.value = ''
submit.on_click(respond)
print('Diabetes Information Assistant')
print('Educational information only, not a diagnosis or treatment service.')
display(widgets.VBox([query_box,submit,output]))

Test the assistant with a query belonging to each defined intent, a differently worded query, an unsupported healthcare query, a low-confidence query and an emergency-related query.

In [ ]:
test_queries = ['thirst', 'type 1 diabetes', 'diabetes diet', 'walking exercise', 'blood glucose test', 'insulin treatment', 'diabetic kidney disease', 'prevent type 2 diabetes', 'What kind of walking exercise is useful?', 'How do I treat malaria?', 'health', 'I have chest pain and difficulty breathing']
test_queries[-2] = 'medicine exercise'
for query in test_queries:
    print('You:',query)
    print('Assistant:',assistant(query))
    print()